# Phase 1E — Pre-game / In-game 통일 베이스라인 비교

**두 예측 시점의 공통 모델:** DNN, Logistic Regression, Random Forest, Histogram Gradient Boosting, XGBoost.  
**In-game 추가 모델:** Empirical Win Expectancy, Leading-Team Rule.

- 기존 학습·검증·테스트 분할 및 DNN 테스트 예측값은 **변경하지 않습니다**.
- 새 앙상블 모델은 **학습셋에서 적합**, **검증셋에서 Brier Score로 설정 선택**, **테스트셋에서 최종 평가**합니다.
- 다만 기존 테스트셋 결과를 이미 확인하였으므로, 이 분석은 **추가 비교 분석**이며 완전히 새로운 외부 검증은 아닙니다.
- In-game의 복수 관측치는 서로 독립적이지 않으므로 불확실성 평가는 **경기 단위 클러스터 부트스트랩**을 사용합니다.


In [ ]:
!pip -q install "xgboost>=2.0,<4" scikit-learn pandas matplotlib

## 1. 이전 단계 ZIP 파일 세 개 업로드

In [ ]:
import io, json, os, zipfile, warnings
from pathlib import Path
import numpy as np, pandas as pd
from google.colab import files
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss, log_loss
from xgboost import XGBClassifier

print('다음 ZIP 세 개를 함께 업로드하세요: phase1c_output.zip, phase1c2_output.zip, phase1d_output.zip')
uploaded = files.upload()
required = ['phase1c_output.zip', 'phase1c2_output.zip', 'phase1d_output.zip']
missing = [f for f in required if not Path(f).exists() and f not in uploaded]
assert not missing, f'필수 파일 누락: {missing}'
zips = {name:zipfile.ZipFile(name) for name in required}
def read_csv(zipname, member):
    return pd.read_csv(io.BytesIO(zips[zipname].read(member)))

pre = read_csv('phase1c_output.zip', 'pregame_features_cleaned_2430.csv')
pre_old = read_csv('phase1c_output.zip', 'pregame_test_predictions.csv')
pre_dnn = read_csv('phase1c2_output.zip', 'selected_test_predictions.csv')
in_game = read_csv('phase1d_output.zip', 'ingame_snapshots_2430.csv')
in_old = read_csv('phase1d_output.zip', 'ingame_test_predictions.csv')
part1 = read_csv('phase1c_output.zip','game_partitions.csv')
part2 = read_csv('phase1d_output.zip','game_partitions.csv')
pd.testing.assert_frame_equal(part1.sort_values('gamePk').reset_index(drop=True), part2.sort_values('gamePk').reset_index(drop=True), check_dtype=False)
assert len(pre)==2430 and pre.gamePk.nunique()==2430
assert len(in_game)==43220 and in_game.gamePk.nunique()==2430
assert all(pre.groupby('gamePk').split.nunique()==1)
assert all(in_game.groupby('gamePk').split.nunique()==1)
print('분할 일치 확인. Pre-game', pre.groupby('split').size().to_dict())
print('In-game',in_game.groupby('split').size().to_dict())

## 2. 평가 및 모델 선택 함수

In [ ]:
def ece10(y, p):
    y=np.asarray(y);p=np.asarray(p); bins=np.linspace(0,1,11)
    b=np.clip(np.digitize(p,bins[1:-1],right=True),0,9)
    return float(sum((b==i).mean()*abs(y[b==i].mean()-p[b==i].mean()) for i in range(10) if (b==i).any()))
def evaluate(y, p):
    y=np.asarray(y,dtype=int);p=np.clip(np.asarray(p,dtype=float),1e-7,1-1e-7)
    return dict(n=len(y),accuracy=float(accuracy_score(y,p>=.5)),auc=float(roc_auc_score(y,p)) if len(np.unique(y))==2 else np.nan,
                brier=float(brier_score_loss(y,p)),logloss=float(log_loss(y,p,labels=[0,1])),ece10=ece10(y,p))

def candidates(seed=2026):
    return {
      'Random Forest':[
        ('RF_leaf5',RandomForestClassifier(n_estimators=250,min_samples_leaf=5,max_features='sqrt',n_jobs=-1,random_state=seed)),
        ('RF_leaf15',RandomForestClassifier(n_estimators=250,min_samples_leaf=15,max_features='sqrt',n_jobs=-1,random_state=seed))],
      'Histogram Gradient Boosting':[
        ('HGB_leaf20',HistGradientBoostingClassifier(max_iter=150,learning_rate=.05,max_leaf_nodes=15,min_samples_leaf=20,l2_regularization=1.,random_state=seed)),
        ('HGB_leaf50',HistGradientBoostingClassifier(max_iter=150,learning_rate=.05,max_leaf_nodes=15,min_samples_leaf=50,l2_regularization=1.,random_state=seed))],
      'XGBoost':[
        ('XGB_depth2',XGBClassifier(n_estimators=220,max_depth=2,learning_rate=.03,subsample=.85,colsample_bytree=.9,reg_lambda=5.,objective='binary:logistic',eval_metric='logloss',tree_method='hist',n_jobs=2,random_state=seed)),
        ('XGB_depth3',XGBClassifier(n_estimators=220,max_depth=3,learning_rate=.03,subsample=.85,colsample_bytree=.9,reg_lambda=5.,objective='binary:logistic',eval_metric='logloss',tree_method='hist',n_jobs=2,random_state=seed))]
    }

def train_baselines(df,features,label):
    assert all(pd.api.types.is_numeric_dtype(df[f]) for f in features), '비수치형 변수가 있습니다'
    assert df[features].notna().all().all(), '입력변수 결측이 있습니다'
    train=df[df.split=='train'];val=df[df.split=='validation'];test=df[df.split=='test']
    Xtr=train[features].to_numpy(float);Xval=val[features].to_numpy(float);Xte=test[features].to_numpy(float)
    ytr=train.target.to_numpy(int);yv=val.target.to_numpy(int)
    scaler=StandardScaler().fit(Xtr)
    lr=LogisticRegression(C=1.0,max_iter=2500,random_state=2026)
    lr.fit(scaler.transform(Xtr),ytr)
    predictions={'Logistic Regression':lr.predict_proba(scaler.transform(Xte))[:,1]}
    val_records=[dict(task=label,model='Logistic Regression',candidate='LR_fixed',**evaluate(yv,lr.predict_proba(scaler.transform(Xval))[:,1]))]
    selection={}
    for model,options in candidates().items():
        scored=[]
        for tag,est in options:
            est.fit(Xtr,ytr)
            pr=est.predict_proba(Xval)[:,1]
            m=evaluate(yv,pr)
            val_records.append(dict(task=label,model=model,candidate=tag,**m))
            scored.append((m['brier'],-m['auc'],tag,est))
        best=min(scored,key=lambda z:(z[0],z[1]))
        selection[model]=best[2]
        predictions[model]=best[3].predict_proba(Xte)[:,1]
        print(label, model, 'selected:',best[2], 'validation Brier:',round(best[0],5))
    return test.copy(),predictions,pd.DataFrame(val_records),selection

PRE_FEATURES=['home_win_pct','away_win_pct','win_pct_diff','home_run_diff','away_run_diff','run_diff_diff',
'home_runs_per_game','away_runs_per_game','home_ra_per_game','away_ra_per_game','home_recent_form','away_recent_form',
'form_diff','home_sp_era','away_sp_era','sp_era_diff','home_sp_whip','away_sp_whip','sp_whip_diff',
'home_sp_k9','away_sp_k9','sp_k9_diff']
IN_FEATURES=['inning','is_bottom','score_diff','home_runs','away_runs','home_hits','away_hits','abs_score_diff']

## 3. Pre-game 베이스라인 학습 / 기존 C1 DNN 합치기

In [ ]:
pre_test,pre_pred,pre_val,pre_select=train_baselines(pre,PRE_FEATURES,'Pre-game')
pre_test=pre_test[['gamePk','date','target']].reset_index(drop=True)
pre_dnn=pre_dnn[['gamePk','target','prob_selected_dnn']]
assert pre_dnn.gamePk.is_unique and set(pre_dnn.gamePk)==set(pre_test.gamePk)
pre_test=pre_test.merge(pre_dnn,on='gamePk',validate='one_to_one',suffixes=('','_dnn'))
assert (pre_test.target==pre_test.target_dnn).all()
pre_test=pre_test.drop(columns='target_dnn')
# train_baselines returns predictions in df test row order, merge preserves left order
pre_test['DNN']=pre_test.pop('prob_selected_dnn')
for k,v in pre_pred.items():pre_test[k]=v
pre_metrics=pd.DataFrame([dict(model=m,**evaluate(pre_test.target,pre_test[m])) for m in ['DNN','Logistic Regression','Random Forest','Histogram Gradient Boosting','XGBoost']])
print(pre_metrics.to_string(index=False))

## 4. In-game 베이스라인 학습 / 기존 DNN 및 상황 기반 베이스라인 합치기

In [ ]:
in_test,in_pred,in_val,in_select=train_baselines(in_game,IN_FEATURES,'In-game')
KEY=['gamePk','inning','is_bottom']
assert not in_test.duplicated(KEY).any() and not in_old.duplicated(KEY).any()
oldcols=['prob_dnn','prob_empirical_win_expectancy','prob_leading_team_rule']
in_test=in_test.merge(in_old[KEY+['target']+oldcols],on=KEY,how='left',validate='one_to_one',suffixes=('','_old'))
assert len(in_test)==6658 and in_test[oldcols].notna().all().all()
assert (in_test.target==in_test.target_old).all()
in_test=in_test.drop(columns='target_old').reset_index(drop=True)
in_test=in_test.rename(columns={'prob_dnn':'DNN','prob_empirical_win_expectancy':'Empirical Win Expectancy',
'prob_leading_team_rule':'Leading-Team Rule'})
for k,v in in_pred.items():in_test[k]=v
inmodels=['DNN','Logistic Regression','Random Forest','Histogram Gradient Boosting','XGBoost',
          'Empirical Win Expectancy','Leading-Team Rule']
in_metrics=pd.DataFrame([dict(model=m,**evaluate(in_test.target,in_test[m])) for m in inmodels])
print(in_metrics.to_string(index=False))

## 5. 기존 Logistic Regression 예측의 일치도 점검

이번 LR은 통일된 재학습 방식으로 다시 적합하므로 이전 ZIP의 LR 예측과 소폭 차이가 있을 수 있습니다. 기존 DNN 및 In-game 전용 두 비교모델의 예측값은 **원래 출력 그대로 재사용**했습니다.

In [ ]:
old_pre_lr=pre_old[['gamePk','prob_logistic_regression']]
pre_ck=pre_test[['gamePk','Logistic Regression']].merge(old_pre_lr,on='gamePk',validate='one_to_one')
print('Pre-game LR 최대 예측 차이:',float(np.max(np.abs(pre_ck['Logistic Regression']-pre_ck.prob_logistic_regression))))
old_in_lr=in_old[KEY+['prob_logistic_regression']]
in_ck=in_test[KEY+['Logistic Regression']].merge(old_in_lr,on=KEY,validate='one_to_one')
print('In-game LR 최대 예측 차이:',float(np.max(np.abs(in_ck['Logistic Regression']-in_ck.prob_logistic_regression))))

## 6. 이닝별 성능 및 경기 단위 부트스트랩

In [ ]:
by_inning=[]
for inning,g in in_test.groupby('inning'):
    for model in inmodels:
        by_inning.append(dict(inning=int(inning),model=model,**evaluate(g.target,g[model])))
by_inning=pd.DataFrame(by_inning)

def game_cluster_bootstrap(df, model_a, model_b, metric='brier', B=1000, seed=2026):
    if metric!='brier': raise ValueError('현재는 Brier 점수만 지원합니다')
    rng=np.random.default_rng(seed)
    # Per-game squared-error sums: exact weighted aggregate and efficient cluster resampling.
    tmp=df[['gamePk','target',model_a,model_b]].copy()
    tmp['sse_a']=(tmp[model_a]-tmp.target)**2
    tmp['sse_b']=(tmp[model_b]-tmp.target)**2
    g=tmp.groupby('gamePk').agg(n=('target','size'),sa=('sse_a','sum'),sb=('sse_b','sum'))
    n=g.n.to_numpy(); sa=g.sa.to_numpy(); sb=g.sb.to_numpy()
    idx=rng.integers(0,len(g),size=(B,len(g)))
    diff=(sa[idx].sum(axis=1)-sb[idx].sum(axis=1))/n[idx].sum(axis=1)
    return float(np.percentile(diff,2.5)),float(np.percentile(diff,97.5))

boot=[]
for label,df,models in [('Pre-game',pre_test,['Logistic Regression','Random Forest','Histogram Gradient Boosting','XGBoost']),
                        ('In-game',in_test,['Logistic Regression','Random Forest','Histogram Gradient Boosting','XGBoost','Empirical Win Expectancy','Leading-Team Rule'])]:
    for m in models:
        ci=game_cluster_bootstrap(df,'DNN',m,metric='brier',B=600)
        boot.append(dict(task=label,contrast='DNN minus '+m,metric='Brier',
                         point_diff=evaluate(df.target,df.DNN)['brier']-evaluate(df.target,df[m])['brier'],
                         ci_low=ci[0],ci_high=ci[1]))
boot=pd.DataFrame(boot)
print(boot.to_string(index=False))

## 7. ZIP 출력

In [ ]:
from datetime import datetime,timezone
out=Path('phase1e_baseline_output');out.mkdir(exist_ok=True)
pre_metrics.to_csv(out/'pregame_comparison_metrics.csv',index=False)
in_metrics.to_csv(out/'ingame_comparison_metrics.csv',index=False)
pre_val.to_csv(out/'pregame_validation_candidates.csv',index=False)
in_val.to_csv(out/'ingame_validation_candidates.csv',index=False)
pre_test.to_csv(out/'pregame_comparison_test_predictions.csv',index=False)
in_test.to_csv(out/'ingame_comparison_test_predictions.csv',index=False)
by_inning.to_csv(out/'ingame_comparison_by_inning.csv',index=False)
boot.to_csv(out/'game_cluster_bootstrap_brier.csv',index=False)
audit={'files_used':required,'pre_test_games':int(pre_test.gamePk.nunique()),
       'in_test_games':int(in_test.gamePk.nunique()),'in_test_snapshots':int(len(in_test)),
       'pregame_selected_candidates':pre_select,'ingame_selected_candidates':in_select,
       'selection':'minimal prespecified candidate grid, validation Brier; tie breaker AUC',
       'heldout_test':'2025-09 chronological split, already viewed in previous phases; no tuning to test',
       'pre_dnn_source':'selected_test_predictions.csv from Phase 1C2',
       'in_dnn_and_domain_baselines':'ingame_test_predictions.csv from Phase 1D',
       'bootstrap':'paired resampling by gamePk, 600 replicates for Brier differences',
       'important_limitations':['Repeated half-inning records are correlated within games',
         'Selection performed on same fixed validation period',
         'September holdout has been inspected in prior model iterations',
         'Pre-game pitcher ID pre-announcement status remains to be verified']}
(out/'comparison_audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')
zipname='phase1e_baseline_output.zip'
with zipfile.ZipFile(zipname,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in out.iterdir():z.write(p,p.name)
print('결과물:',zipname, '크기:',Path(zipname).stat().st_size)
files.download(zipname)